# 🎯 Modul 07 Hands-on: Practical Vision Systems (Object Detection)

Di hands-on ini, Anda akan mempraktikkan komponen inti deteksi objek:
1. **Menghitung dan Memvisualisasikan IoU (Intersection over Union)**.
2. **Algoritma NMS (Non-Maximum Suppression)** untuk mengeliminasi kotak ganda.
3. **Inference Deteksi Objek Modern**: Menggunakan model deteksi objek pre-trained dari Torchvision.
4. **Visualisasi Bounding Box**: Menggambar kotak, label kelas, dan skor keyakinan.

In [ ]:
import torch
import torchvision
import torchvision.transforms.functional as F
from torchvision.models.detection import ssdlite320_mobilenet_v3_large, SSDLite320_MobileNet_V3_Large_Weights
from torchvision.ops import nms
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"🚀 Device aktif: {device}")

## 1. Memahami IoU (Intersection over Union) Secara Visual
Format kotak Pascal VOC: `[x_min, y_min, x_max, y_max]`.

In [ ]:
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    inter_w = max(0, x2 - x1)
    inter_h = max(0, y2 - y1)
    inter_area = inter_w * inter_h
    
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = area1 + area2 - inter_area
    
    return inter_area / union_area if union_area > 0 else 0

box_gt = [50, 50, 200, 200]    # Ground Truth
box_pred = [80, 80, 220, 220]  # Prediksi Model
iou_val = compute_iou(box_gt, box_pred)

fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(0, 300); ax.set_ylim(0, 300); ax.invert_yaxis()

# Gambar kedua kotak
rect_gt = patches.Rectangle((box_gt[0], box_gt[1]), box_gt[2]-box_gt[0], box_gt[3]-box_gt[1], linewidth=2.5, edgecolor="green", facecolor="none", label="Ground Truth")
rect_pred = patches.Rectangle((box_pred[0], box_pred[1]), box_pred[2]-box_pred[0], box_pred[3]-box_pred[1], linewidth=2.5, edgecolor="red", linestyle="--", facecolor="none", label="Model Prediction")
ax.add_patch(rect_gt)
ax.add_patch(rect_pred)

plt.title(f"Visualisasi IoU: {iou_val:.3f} (Overlap Area)", fontsize=12, fontweight="bold")
plt.legend(fontsize=10)
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

## 2. Non-Maximum Suppression (NMS): Menghilangkan Kotak Ganda
Model mentah sering memprediksi 4–6 kotak tumpang tindih untuk satu objek yang sama. NMS menyaringnya secara otomatis.

In [ ]:
# Simulasikan 5 kotak deteksi di sekitar satu objek yang sama dengan skor berbeda
boxes = torch.tensor([
    [50., 50., 200., 200.],   # Kotak 1: Skor 0.95 (Paling akurat)
    [52., 48., 205., 198.],   # Kotak 2: Skor 0.88 (Duplikat)
    [48., 55., 195., 202.],   # Kotak 3: Skor 0.72 (Duplikat)
    [250., 50., 350., 200.],  # Kotak 4: Skor 0.90 (Objek lain yang sah!)
    [55., 60., 210., 205.]    # Kotak 5: Skor 0.60 (Duplikat)
])
scores = torch.tensor([0.95, 0.88, 0.72, 0.90, 0.60])

# Jalankan NMS dari Torchvision dengan threshold IoU = 0.5
keep_indices = nms(boxes, scores, iou_threshold=0.5)

print(f"Total kotak awal  : {len(boxes)}")
print(f"Kotak tersisa (NMS): {len(keep_indices)} (Index terpilih: {keep_indices.tolist()})")
print("💡 Berhasil! Kotak duplikat berhasil disaring dan tersisa hanya 2 objek terpisah!")

## 3. Running Real-Time Pretrained Detector (SSDLite MobileNetV3)
Kita gunakan model detektor ringan yang dilatih pada 80 kelas COCO dataset.

In [ ]:
# Load model deteksi objek pre-trained
weights = SSDLite320_MobileNet_V3_Large_Weights.DEFAULT
model_detector = ssdlite320_mobilenet_v3_large(weights=weights).to(device)
model_detector.eval()

coco_classes = weights.meta["categories"]
print(f"Model siap mendeteksi {len(coco_classes)} kategori COCO (Person, Car, Dog, Bicycle, dll)")

## 4. Simulasi Inferensi & Visualisasi Bounding Box

In [ ]:
# Buat gambar simulasi sintetis dengan 2 objek (persegi warna)
img_test = torch.zeros(1, 3, 320, 320)
img_test[:, 0, 50:180, 50:180] = 0.8   # Kotak merah (simulasi objek)
img_test[:, 1, 100:250, 200:300] = 0.9  # Kotak hijau (simulasi objek 2)

# Inferensi
with torch.no_grad():
    predictions = model_detector(img_test.to(device))[0]

pred_boxes = predictions["boxes"].cpu().numpy()
pred_scores = predictions["scores"].cpu().numpy()
pred_labels = predictions["labels"].cpu().numpy()

print(f"Total kandidat deteksi ditemukan: {len(pred_boxes)}")
print("Filter deteksi dengan keyakinan > 20%:")
for i, score in enumerate(pred_scores):
    if score > 0.20:
        label_name = coco_classes[pred_labels[i]]
        box = pred_boxes[i]
        print(f"-> [{label_name}] Confidence: {score*100:.1f}% | Box: {box.astype(int)}")

print("\n🎉 Modul 07 selesai! Anda sekarang memahami mekanisme Bounding Box, IoU, NMS, dan pipeline deteksi objek!")